<h1 style="color:DodgerBlue">Индивидальный проект</h1>

## Название проекта

Вариант 9. Система заказов товаров и услуг


## Описание проекта

Базовый класс Order хранит OrderId, CreationDate и TotalAmount, реализует AddItem(Item item), RemoveItem(Item item) и CalculateTotal(). Item представляет позицию с названием, ценой и количеством.

- OnlineOrder хранит CustomerEmail и способ доставки. Переопределённый AddItem выводит сведения о доставке.
- PhysicalOrder хранит DeliveryAddress. Переопределённый RemoveItem выводит сведения о возврате удалённого товара.
- SpecializedOrder хранит SpecialConditions. Переопределённый CalculateTotal применяет скидку, указанную в специальных условиях.

Коллекция позиций скрыта внутри Order — инкапсуляция. Три производных класса показывают наследование. Вызовы через массив Order[] демонстрируют полиморфизм. Каждый производный класс содержит собственные свойства и дополнительные методы.

Дополнительное задание Sprint1: конструкторы всех классов и свойства с геттерами и сеттерами. Сеттеры проверяют значения, изменение скидки пересчитывает сумму. Метод TransferItemTo организует взаимодействие двух заказов: вызывает AddItem другого заказа и удаляет позицию из текущего. Order также взаимодействует с Item через GetCost().


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;

public class Item
{
    public string Name { get; }
    public decimal Price { get; }
    public int Quantity { get; }
    public Item(string name, decimal price, int quantity)
    {
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Название пустое");
        if (price < 0) throw new ArgumentOutOfRangeException(nameof(price));
        if (quantity <= 0) throw new ArgumentOutOfRangeException(nameof(quantity));
        Name = name;
        Price = price;
        Quantity = quantity;
    }
    public decimal GetCost() { return Price * Quantity; }
}

public class Order
{
    private readonly List<Item> items = new List<Item>();
    public int OrderId { get; }
    public DateTime CreationDate { get; }
    public decimal TotalAmount { get; protected set; }
    public Order(int orderId, DateTime creationDate)
    {
        if (orderId <= 0) throw new ArgumentOutOfRangeException(nameof(orderId));
        OrderId = orderId;
        CreationDate = creationDate;
    }
    public virtual decimal CalculateTotal()
    {
        decimal total = 0;
        foreach (Item item in items) total += item.GetCost();
        TotalAmount = total;
        return TotalAmount;
    }
    public virtual void AddItem(Item item)
    {
        if (item == null) throw new ArgumentNullException(nameof(item));
        items.Add(item);
        CalculateTotal();
        Console.WriteLine($"Заказ {OrderId}: добавлен {item.Name}");
    }
    public virtual void RemoveItem(Item item)
    {
        if (item == null) throw new ArgumentNullException(nameof(item));
        if (!items.Remove(item)) { Console.WriteLine("Товар отсутствует в заказе"); return; }
        CalculateTotal();
        Console.WriteLine($"Заказ {OrderId}: удалён {item.Name}");
    }
    protected bool ContainsItem(Item item) { return items.Contains(item); }
    public void Show()
    {
        Console.WriteLine($"Заказ {OrderId} от {CreationDate:dd.MM.yyyy}");
        foreach (Item item in items)
            Console.WriteLine($"{item.Name}: {item.Price} x {item.Quantity} = {item.GetCost()}");
        Console.WriteLine($"Итого: {CalculateTotal():F2}");
    }
    public void TransferItemTo(Order target, Item item)
    {
        if (target == null) throw new ArgumentNullException(nameof(target));
        if (item == null) throw new ArgumentNullException(nameof(item));
        if (ReferenceEquals(this, target)) throw new ArgumentException("Выберите другой заказ");
        if (!ContainsItem(item)) { Console.WriteLine("Нечего переносить"); return; }
        target.AddItem(item);
        RemoveItem(item);
        Console.WriteLine($"Товар перенесён из заказа {OrderId} в заказ {target.OrderId}");
    }
}

public class OnlineOrder : Order
{
    private string customerEmail;
    public string CustomerEmail
    {
        get { return customerEmail; }
        set
        {
            if (string.IsNullOrWhiteSpace(value)) throw new ArgumentException("Значение не может быть пустым");
            customerEmail = value;
        }
    }
    private string deliveryMethod;
    public string DeliveryMethod
    {
        get { return deliveryMethod; }
        set
        {
            if (string.IsNullOrWhiteSpace(value)) throw new ArgumentException("Значение не может быть пустым");
            deliveryMethod = value;
        }
    }
    public OnlineOrder(int id, DateTime date, string email, string deliveryMethod)
        : base(id, date)
    {
        CustomerEmail = email;
        DeliveryMethod = deliveryMethod;
    }
    public override void AddItem(Item item)
    {
        base.AddItem(item);
        Console.WriteLine($"Способ доставки: {DeliveryMethod}; email клиента: {CustomerEmail}");
    }
    public void ShowDelivery() { Console.WriteLine($"Доставка: {DeliveryMethod}"); }
}

public class PhysicalOrder : Order
{
    private string deliveryAddress;
    public string DeliveryAddress
    {
        get { return deliveryAddress; }
        set
        {
            if (string.IsNullOrWhiteSpace(value)) throw new ArgumentException("Значение не может быть пустым");
            deliveryAddress = value;
        }
    }
    public PhysicalOrder(int id, DateTime date, string address) : base(id, date)
    {
        DeliveryAddress = address;
    }
    public override void RemoveItem(Item item)
    {
        bool existed = ContainsItem(item);
        base.RemoveItem(item);
        if (existed) Console.WriteLine($"Возврат товара {item.Name}; адрес доставки: {DeliveryAddress}");
    }
    public void ShowAddress() { Console.WriteLine($"Адрес доставки: {DeliveryAddress}"); }
}

public class SpecializedOrder : Order
{
    public string SpecialConditions { get; private set; }
    private decimal discountPercent;
    public decimal DiscountPercent
    {
        get { return discountPercent; }
        set
        {
            if (value < 0 || value > 100) throw new ArgumentOutOfRangeException(nameof(value));
            discountPercent = value;
            SpecialConditions = $"Скидка {value}% на все позиции";
            CalculateTotal();
        }
    }
    public SpecializedOrder(int id, DateTime date, decimal discount) : base(id, date)
    {
        DiscountPercent = discount;
    }
    public override decimal CalculateTotal()
    {
        // Считаем от исходной суммы, поэтому скидка не накапливается при повторном вызове.
        TotalAmount = decimal.Round(base.CalculateTotal() * (1 - DiscountPercent / 100m), 2);
        return TotalAmount;
    }
    public void ShowConditions() { Console.WriteLine(SpecialConditions); }
}

DateTime date = new DateTime(2026, 9, 24);
Item notebook = new Item("Тетрадь", 80m, 3);
Item pen = new Item("Ручка", 50m, 2);
OnlineOrder online = new OnlineOrder(1, date, "student@example.com", "Курьер");
PhysicalOrder physical = new PhysicalOrder(2, date, "Тюмень, ул. Республики, 1");
SpecializedOrder specialized = new SpecializedOrder(3, date, 10m);

// Вызовы через Order демонстрируют полиморфизм.
Order[] orders = { online, physical, specialized };
foreach (Order order in orders)
{
    order.AddItem(notebook);
    order.AddItem(pen);
    order.Show();
}
Order physicalAsOrder = physical;
physicalAsOrder.RemoveItem(pen);
physicalAsOrder.Show();
online.ShowDelivery();
physical.ShowAddress();
specialized.ShowConditions();
online.DeliveryMethod = "Пункт выдачи";
online.TransferItemTo(physical, pen);
specialized.DiscountPercent = 15m;
specialized.ShowConditions();
foreach (Order order in orders) order.Show();
